# 08 · Create a collection: size and distance
Level L2 · Part 2 · builds on lesson 07 (and the metrics from lesson 03)

## Goal
By the end of this lesson you can create the `recipes` collection with the right vector size and
distance metric, read both settings back, and explain why you chose them.

## The idea
A **collection** asks for two settings before it accepts a single point. The **vector size** is
how many numbers every vector has; it must match the embedding model exactly, because the
database refuses any vector of a different length. The **distance metric** is the rule used to
score a query against each stored vector: Cosine, Dot, Euclid or Manhattan (lesson 03 taught the
first three by hand). **Manhattan** is the fourth: add up the absolute difference on each
dimension, so `[0.3, 0.1]` to `[0.3, 0.4]` is 0 + 0.3 = 0.3. Both settings are fixed when the
collection is created, like the bay width of a parking garage.

## Picture

```mermaid
flowchart LR
    M["embedding model<br/>all-MiniLM-L6-v2"] -- "vectors of 384 numbers,<br/>unit length" --> S["size = 384"]
    M -- "compared by angle<br/>(cosine)" --> D["distance = Cosine"]
    S --> C[("collection 'recipes'<br/>size 384, Cosine")]
    D --> C
    C -- "accepts only 384-number vectors,<br/>scores every query with Cosine" --> Q["query_points"]
```

Both settings come from the model, not from taste: the size is read from its output, the
distance from how its vectors are meant to be compared.

## Step by step
### Step 1 · Read the size off the data
Lesson 07 typed `size=384`. Better: read it from the embeddings themselves, so the collection can
never disagree with the model that made them. We also check the vectors' lengths, which matters
for the distance choice later.

In [1]:
import numpy as np
import pandas as pd

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb = saved["doc_emb"]
print("model:", saved["model"], "  doc_emb shape:", doc_emb.shape, doc_emb.dtype)
VECTOR_SIZE = doc_emb.shape[1]
print("vector size:", VECTOR_SIZE)
print("lengths of the first 3 vectors:", np.round(np.linalg.norm(doc_emb[:3], axis=1), 4))
lengths = np.linalg.norm(doc_emb, axis=1)
print("min/max length of all 50:", round(float(lengths.min()), 4), round(float(lengths.max()), 4))

model: sentence-transformers/all-MiniLM-L6-v2   doc_emb shape: (50, 384) float32
vector size: 384
lengths of the first 3 vectors: [1. 1. 1.]
min/max length of all 50: 1.0 1.0


50 vectors of 384 numbers each, so `VECTOR_SIZE = 384`. All 50 lengths are 1.0: the model returns
unit vectors (lesson 05). Keep that in mind for Step 5.

### Step 2 · Create the `recipes` collection
We open a local-mode client on this lesson's own folder (lesson 07) and create the collection
only if it is not there yet. `collection_exists` makes the cell safe to run twice.

In [2]:
import shutil, warnings
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

DB_PATH = "../data/cache/qdrant_08"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)

if not client.collection_exists("recipes"):
    client.create_collection(
        collection_name="recipes",
        vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
    )
print("collections:", [c.name for c in client.get_collections().collections])

collections: ['recipes']


Now read the two settings back. A program that writes to a collection someone else created
should check them first instead of assuming.

In [3]:
params = client.get_collection("recipes").config.params.vectors
print("size:    ", params.size)
print("distance:", params.distance)
print("points:  ", client.count("recipes").count)

size:     384
distance: Cosine
points:   0


`size: 384`, `distance: Cosine`, and zero points: an empty, correctly shaped
collection. Lesson 09 fills it.

### Step 3 · The size is enforced
The collection checks every vector's length. Try storing the 2-D Tomato soup vector from
lessons 02 and 03 (`[sweet, spicy]`) into the 384-number collection:

In [4]:
try:
    client.upsert("recipes", points=[models.PointStruct(id=1, vector=[0.3, 0.1])])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Wrong input: Vector dimension error: expected dim: 384, got 2 for vector ''


It prints `ValueError: Wrong input: Vector dimension error: expected dim: 384, got 2 for vector ''`.
The `''` is the name of the vector: a collection created with one plain `VectorParams` has a
single unnamed vector (lesson 22 adds named ones). Nothing was stored:

In [5]:
print("points after the failed upsert:", client.count("recipes").count)
print("first recipe vector has", len(doc_emb[0]), "numbers, the collection wants", params.size)
print("would it fit?", len(doc_emb[0]) == params.size)

points after the failed upsert: 0
first recipe vector has 384 numbers, the collection wants 384
would it fit? True


### Step 4 · Four distances, four different rankings
The distance decides what "closest" means. To see it, we reuse lesson 03's toy: five recipes
scored `[sweet, spicy]` and the query "mildly spicy and a bit sweet" `[0.3, 0.4]`. We build one
tiny 2-D collection per metric in an in-memory client (`":memory:"`, nothing saved) and ask each
for its top 3. `Distance` is an enum, so we can loop over all four.

In [6]:
toy = {                     # [sweet, spicy], same hand scores as lessons 02 and 03
    "Tomato soup": [0.3, 0.1], "Mango salsa": [0.6, 0.8], "Chocolate brownies": [0.95, 0.0],
    "Caesar salad": [0.1, 0.1], "Lemon sorbet": [0.8, 0.0],
}
toy_query = [0.3, 0.4]
lab = QdrantClient(":memory:")
for metric in models.Distance:
    lab.create_collection(f"toy_{metric.value}", vectors_config=models.VectorParams(size=2, distance=metric))
    lab.upsert(f"toy_{metric.value}", points=[
        models.PointStruct(id=i, vector=v, payload={"title": name}) for i, (name, v) in enumerate(toy.items())])
    hits = lab.query_points(f"toy_{metric.value}", query=toy_query, limit=3).points
    print(f"{metric.value:10}", [(p.payload["title"], round(p.score, 3)) for p in hits])

Cosine     [('Mango salsa', 1.0), ('Caesar salad', 0.99), ('Tomato soup', 0.822)]
Euclid     [('Tomato soup', 0.3), ('Caesar salad', 0.361), ('Mango salsa', 0.5)]
Dot        [('Mango salsa', 0.5), ('Chocolate brownies', 0.285), ('Lemon sorbet', 0.24)]
Manhattan  [('Tomato soup', 0.3), ('Caesar salad', 0.5), ('Mango salsa', 0.7)]


Read the scores carefully:
- **Cosine** and **Dot** are similarities: bigger is closer, so the list is in *descending*
  order. Mango salsa scores 1.0 under Cosine: it points in exactly the query's direction.
- **Euclid** and **Manhattan** are distances: smaller is closer, so the list is in *ascending*
  order and the score is the distance itself (Tomato soup is 0.3 away).
- The winners differ. Cosine and Dot pick Mango salsa (same direction); Euclid and Manhattan pick
  Tomato soup (nearest point on the map). Dot ranks Chocolate brownies second because its long
  vector inflates the score, the effect lesson 03 warned about.

Same data, same query, different answers: the metric is part of the meaning of "similar".

### Step 5 · Cosine normalises what you store
Qdrant's docs say Cosine is computed as a dot product over normalised vectors, and vectors are
normalised on upload. Ask the toy Cosine collection for its stored vectors:

In [7]:
stored = lab.retrieve("toy_Cosine", ids=[0, 1, 2], with_vectors=True)
for p in stored:
    v = np.array(p.vector)
    print(f"{p.payload['title']:20} sent {toy[p.payload['title']]}  stored {np.round(v, 3)}  length {np.linalg.norm(v):.3f}")

Tomato soup          sent [0.3, 0.1]  stored [0.949 0.316]  length 1.000
Mango salsa          sent [0.6, 0.8]  stored [0.6 0.8]  length 1.000
Chocolate brownies   sent [0.95, 0.0]  stored [1. 0.]  length 1.000


Every stored vector now has length 1.000: Tomato soup's `[0.3, 0.1]` came back as
`[0.949, 0.316]`. The direction is kept, the length is thrown away. So a Cosine collection can
never tell you how long the original vector was; if length carries meaning, Cosine is the wrong
choice.

### Step 6 · With unit vectors, three metrics agree
Our MiniLM vectors are already unit length (Step 1). For unit vectors, cosine equals the dot
product (lesson 03), and Euclidean distance is `sqrt(2 - 2 * cosine)`, which shrinks exactly as
cosine grows. So Cosine, Dot and Euclid should rank the 50 recipes identically. Check it with the
saved query "seafood" from the lesson 06 query cache.

In [8]:
cache = np.load("../data/query_embeddings_minilm.npz")
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))
q = query_cache["seafood"].tolist()
rows = list(recipes.itertuples())
for metric in [models.Distance.COSINE, models.Distance.DOT, models.Distance.EUCLID]:
    name = f"recipes_{metric.value}"
    lab.create_collection(name, vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=metric))
    lab.upsert(name, points=[models.PointStruct(id=int(r.id), vector=doc_emb[i].tolist(), payload={"title": r.title})
                             for i, r in enumerate(rows)])
    hits = lab.query_points(name, query=q, limit=3).points
    print(f"{metric.value:7}", [(p.payload["title"], round(p.score, 3)) for p in hits])

Cosine  [('Grilled salmon', 0.502), ('Sushi rolls', 0.489), ('Fish tacos', 0.473)]
Dot     [('Grilled salmon', 0.502), ('Sushi rolls', 0.489), ('Fish tacos', 0.473)]
Euclid  [('Grilled salmon', 0.998), ('Sushi rolls', 1.011), ('Fish tacos', 1.027)]


Same three recipes, same order. Cosine and Dot give identical scores (0.502, 0.489, 0.473, as in
lesson 07). Euclid gives distances. Confirm the formula on the top hit:

In [9]:
cos_top = 0.502
print("sqrt(2 - 2 * 0.502) =", round(float(np.sqrt(2 - 2 * cos_top)), 3))
top = lab.query_points("recipes_Euclid", query=q, limit=1).points[0]
print("Euclid score from Qdrant:", round(top.score, 3), "for", top.payload["title"])

sqrt(2 - 2 * 0.502) = 0.998
Euclid score from Qdrant: 0.998 for Grilled salmon


0.998 both ways. So for a model that outputs unit vectors, the choice between Cosine, Dot and
Euclid changes the score scale, not the ranking.

### Choosing the distance
The usual pick is **Cosine**: it is the similarity sentence-transformers uses for this model by
default, the scores read naturally (1 = same direction), and it stays correct even if a vector
arrives un-normalised. Choose **Dot** only when the model says so (some models deliberately
encode importance in vector length), and **Euclid** or **Manhattan** when your vectors are real
coordinates rather than embeddings.

| Your vectors | Distance | Why |
| --- | --- | --- |
| Text embeddings from a sentence-transformers model (unit length) | Cosine | the model compares by angle; safe if a vector is not normalised |
| A model whose card says "dot product" / maximum inner product | Dot | the length carries meaning; Cosine would erase it |
| Measured coordinates (map positions, sensor readings) | Euclid | straight-line distance is the real-world meaning |
| Coordinates where each axis is counted separately (grid steps) | Manhattan | sum of per-axis differences |

### Step 7 · Size and distance are fixed at creation
Suppose we change our minds and want Dot for `recipes`. There is no setting to switch: the
per-vector update (`VectorParamsDiff`) only covers things like index and storage options, and
rejects `size` or `distance` before anything is sent.

In [10]:
try:
    client.update_collection("recipes", vectors_config={"": models.VectorParamsDiff(distance=models.Distance.DOT)})
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValidationError: 1 validation error for VectorParamsDiff
distance
  Extra inputs are not permitted [type=extra_forbidden, input_value=<Distance.DOT: 'Dot'>, input_type=Distance]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


It prints `ValidationError: 1 validation error for VectorParamsDiff` followed by
`distance: Extra inputs are not permitted`. To change size or distance you create a new
collection and load the points again (lesson 42's aliases make that switch without
downtime). On an empty collection, deleting and recreating is fine:

In [11]:
client.delete_collection("recipes")
client.create_collection(
    collection_name="recipes",
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
)
print(client.get_collection("recipes").config.params.vectors)

size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None


We recreated it with the same, correct settings: size 384, Cosine. The other fields printed as
`None` (index, quantization, storage, datatype) fall back to defaults; Part 4 tunes them.

## What just happened
- Step 1 read `VECTOR_SIZE = 384` from `doc_emb.shape[1]`; all 50 vectors have length 1.0.
- Step 2 created `recipes` (size 384, Cosine) and read both settings back: 0 points.
- Step 3: a 2-number vector failed with `expected dim: 384, got 2`.
- Step 4: Cosine/Dot picked Mango salsa (scores descend), Euclid/Manhattan picked Tomato soup (distances ascend).
- Step 5: Cosine stored `[0.3, 0.1]` as the unit vector `[0.949, 0.316]`.
- Step 6: on unit vectors Cosine, Dot and Euclid gave the same top 3 (Euclid 0.998 = `sqrt(2 - 2 * 0.502)`).
- Step 7: size and distance cannot be changed in place.

## Common mistakes

**1. Copying a size from a different model.** Many tutorials use 768 or 1536 (sizes of larger
models). The collection is created happily; the error comes later, at the first upsert:

In [12]:
try:
    client.create_collection("recipes_768", vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE))
    client.upsert("recipes_768", points=[models.PointStruct(id=1, vector=doc_emb[0].tolist())])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Wrong input: Vector dimension error: expected dim: 768, got 384 for vector ''


It prints `ValueError: Wrong input: Vector dimension error: expected dim: 768, got 384 for vector ''`.
Fix: derive the size from the embeddings (`doc_emb.shape[1]`) or the model
(`model.get_embedding_dimension()`, lesson 05), and drop the wrong collection.

In [13]:
client.delete_collection("recipes_768")
right = models.VectorParams(size=doc_emb.shape[1], distance=models.Distance.COSINE)
print("size taken from the data:", right.size)
print("collections left:", [c.name for c in client.get_collections().collections])

size taken from the data: 384
collections left: ['recipes']


The size check has a blind spot: **bge-small-en-v1.5 also outputs 384 numbers**, so its vectors
would be accepted by this collection without any error, and searches mixing the two models would
return nonsense (lesson 05: vectors from different models cannot be compared). The size guards the
shape, not the model; record the model name somewhere you can check (lesson 09's payload).

**2. Creating a collection that already exists.** Rerunning a "create" cell, or two scripts that
both set up `recipes`:

In [14]:
try:
    client.create_collection("recipes", vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Collection recipes already exists


It prints `ValueError: Collection recipes already exists`. Fix: guard with `collection_exists`
(Step 2), and if it exists, check its settings rather than trusting them:

In [15]:
if client.collection_exists("recipes"):
    existing = client.get_collection("recipes").config.params.vectors
    assert existing.size == VECTOR_SIZE and existing.distance == models.Distance.COSINE, existing
    print("recipes exists with the expected settings:", existing.size, existing.distance.value)

recipes exists with the expected settings: 384 Cosine


**3. Writing the distance in lowercase.** The client validates the name against the enum:

In [16]:
try:
    models.VectorParams(size=VECTOR_SIZE, distance="cosine")
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValidationError: 1 validation error for VectorParams
distance
  Input should be 'Cosine', 'Euclid', 'Dot' or 'Manhattan' [type=enum, input_value='cosine', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


It prints `ValidationError: 1 validation error for VectorParams` and
`Input should be 'Cosine', 'Euclid', 'Dot' or 'Manhattan'`. Fix: use the enum,
`models.Distance.COSINE`, so a typo fails in your editor rather than at run time.

In [17]:
ok = models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
print(ok.size, ok.distance.value)
print("all valid names:", [d.value for d in models.Distance])

384 Cosine
all valid names: ['Cosine', 'Euclid', 'Dot', 'Manhattan']


## Try it
1. **Length matters to Dot.** Add a sixth toy recipe, "Double brownies" `[1.9, 0.0]` (twice
   Chocolate brownies), to fresh 2-D Cosine and Dot collections and query `[0.3, 0.4]`. Where does
   it rank under each metric, and why?
2. **Pick for bge-small.** Using `../data/recipe_embeddings_bge-small.npz`, decide the size and
   distance for a `recipes_bge` collection from the data alone (shape and vector lengths), create
   it in the in-memory `lab` client, and print its settings.

In [18]:
# your code here

Close both clients so the local folder is released.

In [19]:
lab.close()
client.close()
print("closed")

closed


## Key terms
- **vector size** — how many numbers every vector in a collection has; set at creation and checked on every upsert and query; must equal the embedding model's output size.
- **distance metric** — the rule a collection uses to score a query against stored vectors: Cosine, Dot, Euclid or Manhattan; fixed at creation.
- **Manhattan distance** — the sum of the absolute differences on each dimension; smaller is closer; also called L1 distance.
- **in-memory client** — `QdrantClient(":memory:")`: local mode that keeps everything in RAM and saves nothing, handy for experiments.